## Assignment 8

| Member Name      | Part 1 - Install Libraries | Part 2 - Check Data | Part 3 - Create Map  |
|------------------|----------------------------|---------------------|---------------------|
| Luis Felipe Acosta         | Responsible                 |                     |                     |
| Alejandro Ospina        |                            | Responsible          |                     |
| Raúl Amao         |                            |                     | Responsible          |
| Ruth Chávez         |                            |                     |  Responsible                   |
| Manuel Soto         |                            |                     |   Responsible                  |

### Part 1: Installing and Importing Necessary Libraries

In [ ]:
# Part 1: Installing and importing the necessary libraries
#!pip install geopandas rasterio folium

import geopandas as gpd
import rasterio
from rasterio.mask import mask
import folium

### Part 2: Checking the Data

In [ ]:
# Load the shapefile of Peru departments
shapefile_path = '../../_data/LIMITE_DISTRITAL_2020_INEI/INEI_LIMITE_DISTRITAL.shp'
peru_departments = gpd.read_file(shapefile_path)

In [ ]:
# Load the raster file
#raster_path = 'C:/Users/raul3/Downloads/TIF FILES/GHS_BUILT_C_MSZ_E2018_GLOBE_R2023A_54009_10_V1_0_R10_C11.tif'
# Step 2: Check the CRS of the shapefile
print("Shapefile CRS:", peru_departments.crs)

# Step 3: Open the raster and check its CRS
raster_path = 'C:/Users/raul3/Downloads/TIF FILES/GHS_BUILT_C_MSZ_E2018_GLOBE_R2023A_54009_10_V1_0_R10_C11.tif'
with rasterio.open(raster_path) as src:
    print("Raster CRS:", src.crs)

# Step 4: If they differ, reproject the shapefile to the raster's CRS
if peru_departments.crs != src.crs:
    peru_departments = peru_departments.to_crs(src.crs)
    print("Reprojected shapefile CRS:", peru_departments.crs)

In [ ]:
# Step 5: Print the bounds of the shapefile
print("Shapefile bounds:", peru_departments.total_bounds)

# Step 6: Print the bounds of the raster
with rasterio.open(raster_path) as src:
    print("Raster bounds:", src.bounds)

In [ ]:
from shapely.geometry import box

# Create a bounding box from the raster's bounds
with rasterio.open(raster_path) as src:
    raster_bounds = src.bounds
    raster_bbox = box(raster_bounds.left, raster_bounds.bottom, raster_bounds.right, raster_bounds.top)

# Clip the shapefile to the raster bounds
peru_departments_clipped = peru_departments.clip(raster_bbox)

# Now proceed with extracting raster data using peru_departments_clipped

In [ ]:
print(peru_departments.columns)

In [ ]:
print(peru_departments.head())

In [ ]:
# Check the data types of all columns
print(peru_departments.dtypes)

### Part 3: Creating an Interactive Map with GeoJson Layer

In [ ]:
# Part 3: Creating an interactive map with department geometries and popups

# Step 1: Initialize a Folium map centered on Peru
peru_map = folium.Map(location=[-9.19, -75.0152], zoom_start=6)

# Step 2: Add GeoJson layer with popups for 'CCDD', 'NOMBDEP', 'NOMBDIST', and 'UBIGEO'
for _, row in peru_departments.iterrows():
    # Create a popup with the relevant information
    popup_text = (f"CCDD: {row['CCDD']}<br>"
                  f"NOMBDEP: {row['NOMBDEP']}<br>"
                  f"NOMBDIST: {row['NOMBDIST']}<br>"
                  f"UBIGEO: {row['UBIGEO']}")

    # Add the geometry and the popup to the map
    folium.GeoJson(
        row['geometry'],  # Use the geometry of the department
        name=row['NOMBDEP'],  # Department name as the layer name
        popup=folium.Popup(popup_text, max_width=300)  # Create a popup with the data
    ).add_to(peru_map)

# Step 3: Add LayerControl to toggle between layers
folium.LayerControl().add_to(peru_map)

# Step 4: Save the map as an HTML file
output_path = 'group_2_ass_8_2024_2.html'
peru_map.save(output_path)

print(f'Map with popups saved as: {output_path}')
